In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, window, count
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)
from pyspark.sql import Window, functions as F                                                                                   
from pathlib import Path
import sys                                                                                                   
sys.path.append("..") 

In [2]:
spark = SparkSession.builder \
        .appName("KafkaIncrementalBatch") \
        .master("local[*]") \
        .config(
            "spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
        ) \
        .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


transaction_schema = StructType([
    StructField("trans_date_trans_time", StringType(), True),
    StructField("cc_num", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amt", StringType(), True),
    StructField("first", StringType(), True),
    StructField("last", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip", StringType(), True),
    StructField("lat", StringType(), True),
    StructField("long", StringType(), True),
    StructField("city_pop", StringType(), True),
    StructField("job", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("trans_num", StringType(), True),
    StructField("unix_time", StringType(), True),
    StructField("merch_lat", StringType(), True),
    StructField("merch_long", StringType(), True),
    StructField("is_fraud", StringType(), True),
    StructField("merch_zipcode", StringType(), True)
])

Spark version: 4.2.0


In [ ]:
project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "config.py").is_file()),
    Path.cwd(),
)
processed_data_path = project_root / "data" / "processed" / "transactions_parquet"
checkpoint_path = project_root / "data" / "checkpoints" / "transactions_parquet"

# 2. Define the function that processes every micro-batch of 100 rows
def process_and_write_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    window_10m = (
        Window.partitionBy("cc_num")
        .orderBy("unix_time")
        .rangeBetween(-10 * 60, -1)
    )
    window_1h = (
        Window.partitionBy("cc_num")
        .orderBy("unix_time")
        .rangeBetween(-60 * 60, -1)
    )

    cleaned_transactions_df = (
        batch_df
        .withColumn("trans_date_trans_time", F.to_timestamp("trans_date_trans_time", "yyyy-MM-dd HH:mm:ss"))
        .withColumn("dob", F.to_date("dob", "yyyy-MM-dd"))
        .withColumn("amt", F.col("amt").cast("double"))
        .withColumn("lat", F.col("lat").cast("double"))
        .withColumn("long", F.col("long").cast("double"))
        .withColumn("merch_lat", F.col("merch_lat").cast("double"))
        .withColumn("merch_long", F.col("merch_long").cast("double"))
        .withColumn("city_pop", F.col("city_pop").cast("long"))
        .withColumn("unix_time", F.col("unix_time").cast("long"))
        .withColumn("is_fraud", F.col("is_fraud").cast("int"))
        .withColumn("trans_num", F.trim("trans_num"))
        .withColumn("category", F.lower(F.trim("category")))
        .filter(
            F.col("trans_date_trans_time").isNotNull()
            & F.col("trans_num").isNotNull()
            & (F.length("trans_num") > 0)
            & F.col("amt").isNotNull()
            & (F.col("amt") >= 0)
            & F.col("unix_time").isNotNull()
            & F.col("is_fraud").isin(0, 1)
        )
        .dropDuplicates(["trans_num"])
    )

    transactions_df = (
        cleaned_transactions_df
        .withColumn("transaction_hour", F.hour("trans_date_trans_time"))
        .withColumn("transaction_day_of_week", F.dayofweek("trans_date_trans_time"))
        .withColumn("is_weekend", F.dayofweek("trans_date_trans_time").isin(1, 7).cast("int"))
        .withColumn("amt_log1p", F.log1p("amt"))
        .withColumn(
            "distance_km",
            F.round(
                6371.0 * 2 * F.asin(
                    F.sqrt(
                        F.least(
                            F.lit(1.0),
                            F.pow(
                                F.sin(F.radians(F.col("merch_lat") - F.col("lat")) / 2),
                                2,
                            )
                            + F.cos(F.radians("lat"))
                            * F.cos(F.radians("merch_lat"))
                            * F.pow(
                                F.sin(F.radians(F.col("merch_long") - F.col("long")) / 2),
                                2,
                            ),
                        )
                    )
                ),
                3,
            ),
        )
        .withColumn("card_txn_count_10m", F.count("trans_num").over(window_10m))
        .withColumn("card_amount_sum_10m", F.coalesce(F.sum("amt").over(window_10m), F.lit(0.0)))
        .withColumn("card_txn_count_1h", F.count("trans_num").over(window_1h))
        .withColumn("card_amount_sum_1h", F.coalesce(F.sum("amt").over(window_1h), F.lit(0.0)))
    )

    # Write this processed batch to Parquet
    transactions_df.write \
        .mode("append") \
        .parquet(processed_data_path.as_posix())
    
    print(f"Processed and wrote batch {batch_id}")

# 3. Read the stream from Kafka (your source setup)
df_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "transaction-stream")
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 10000)
    .load()
)

parsed_stream_df = (
    df_stream
    .select(
        from_json(
            col("value").cast("string"), # converts value hexadecimal data from kafka to json format
            transaction_schema
        ).alias("data")
    )
    .select("data.*")
)

query = (
    parsed_stream_df.writeStream
    .foreachBatch(process_and_write_batch)
    .option("checkpointLocation", checkpoint_path.as_posix())
    # .trigger(processingTime="10 seconds")
    .start()
)

query.awaitTermination()

Processed and wrote batch 1
Processed and wrote batch 2
Processed and wrote batch 3
Processed and wrote batch 4
Processed and wrote batch 5
Processed and wrote batch 6
Processed and wrote batch 7
Processed and wrote batch 8
Processed and wrote batch 9
Processed and wrote batch 10
Processed and wrote batch 11
Processed and wrote batch 12
Processed and wrote batch 13
Processed and wrote batch 14
Processed and wrote batch 15
Processed and wrote batch 16
Processed and wrote batch 17
Processed and wrote batch 18
Processed and wrote batch 19
Processed and wrote batch 20
Processed and wrote batch 21
Processed and wrote batch 22
Processed and wrote batch 23
Processed and wrote batch 24
Processed and wrote batch 25
Processed and wrote batch 26
Processed and wrote batch 27
Processed and wrote batch 28
Processed and wrote batch 29
Processed and wrote batch 30
Processed and wrote batch 31
Processed and wrote batch 32
Processed and wrote batch 33
Processed and wrote batch 34
Processed and wrote bat